# 04 — Feature Engineering & Spark Scaling


In [1]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
sns.set_theme(style="whitegrid", context="notebook")


In [2]:

from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
from sklearn.preprocessing import StandardScaler
import mlflow
from neuro.bids import validate_bids
from neuro.features import build_feature_table, save_features_parquet, extract_roi_timeseries, get_schaefer_masker
from neuro.spark_utils import get_spark
from neuro.mlflow_utils import start_run

with start_run("04_feature_engineering"):
    report = validate_bids()
    runs = report["runs"]
    spark = get_spark("BladerunnerNeuro_Features")
    runs_pdf = runs[runs["bold_exists"]].copy()
    mlflow.log_param("n_runs", len(runs_pdf))
    print(f"Building features for {len(runs_pdf)} runs")


/home/al/.pyenv/versions/3.10.10/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/22 07:56:44 WARN Utils: Your hostname, frbbmaster002, resolves to a loopback address: 127.0.1.1; using 192.168.1.30 instead (on interface enp11s0)
26/06/22 07:56:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 07:56:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/22 07:56:44 WARN SparkConf: Note that spark.local.dir will be overridden by the value set

Building features for 195 runs


In [3]:

masker = get_schaefer_masker(n_rois=100)

@F.udf(returnType=FloatType())
def roi_mean_scalar(path):
    ts = extract_roi_timeseries(path, masker)
    return float(ts.mean())

files_sdf = spark.createDataFrame(runs_pdf[["subject", "task", "run", "bold_path", "group_short"]])
files_sdf = files_sdf.withColumn("roi_global_mean", roi_mean_scalar(F.col("bold_path")))
spark_means = files_sdf.groupBy("group_short").avg("roi_global_mean").toPandas()
display(spark_means)
sns.barplot(data=spark_means, x="group_short", y="avg(roi_global_mean)", hue="group_short", palette="muted", legend=False)
plt.title("Spark-aggregated ROI global mean by group")
plt.show()


[fetch_atlas_schaefer_2018] Dataset found in /home/al/nilearn_data/schaefer_2018

26/06/22 07:56:52 ERROR Executor: Exception in task 1.0 in stage 0.0 (TID 1) 12]
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 3369, in main
    func, profiler, deserializer, serializer = read_udfs(pickleSer, infile, eval_type)
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 3276, in read_udfs
    read_single_udf(
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1305, in read_single_udf
    f, return_type = read_command(pickleSer, infile)
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker_util.py", line 64, in read_command
    command = serializer._read_with_length(file)
  File "/opt/spark/python/lib/pyspark.zip/pyspark/serializers.py", line 173, in _read_with_length
    return self.loads(obj)
  File "/opt/spark/python/lib/pyspark.zip/pyspark/serializers.py", line 473, in loads
    return cloudpickle.loads(obj, encoding=encoding)
ModuleNotFound

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 3369, in main
    func, profiler, deserializer, serializer = read_udfs(pickleSer, infile, eval_type)
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 3276, in read_udfs
    read_single_udf(
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1305, in read_single_udf
    f, return_type = read_command(pickleSer, infile)
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker_util.py", line 64, in read_command
    command = serializer._read_with_length(file)
  File "/opt/spark/python/lib/pyspark.zip/pyspark/serializers.py", line 173, in _read_with_length
    return self.loads(obj)
  File "/opt/spark/python/lib/pyspark.zip/pyspark/serializers.py", line 473, in loads
    return cloudpickle.loads(obj, encoding=encoding)
ModuleNotFoundError: No module named '

In [ ]:

feat_df = build_feature_table(runs_pdf, n_rois=100)
path = save_features_parquet(feat_df)
mlflow.log_artifact(str(path))
summary = feat_df[["ts_mean", "ts_std", "ts_entropy"]].describe()
display(summary)
sns.pairplot(feat_df, vars=["ts_mean", "ts_std", "ts_entropy"], hue="group_short", corner=True, plot_kws={"alpha": 0.6})
plt.show()
